In [1]:
import dgl.nn as dglnn
from dgl import from_networkx
import torch.nn as nn
import torch as th
import torch.nn.functional as F
import dgl.function as fn
import networkx as nx
import pandas as pd
import socket
import struct
import random
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import category_encoders as ce
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

c:\Users\nq100\miniconda3\DLLs\envs\te-g-sage\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
os.chdir(r'C:\Users\nq100\Downloads\TE-G-SAGE-XAI-workspace\TE-G-SAGE-XAI\netflow')
print("CWD:", os.getcwd())   # ← thêm dòng này để debug

import sys
sys.path.append('datasets')
from data_cleaning import clean_nfunsw_nb15

# load this dataset and clean it
data = pd.read_csv('data/NF-Bot-IoT-v3.csv', skiprows=lambda i: i > 0 and i % 70 != 0)

data = clean_nfunsw_nb15(data)

CWD: C:\Users\nq100\Downloads\TE-G-SAGE-XAI-workspace\TE-G-SAGE-XAI\netflow
[clean] Step 1/3: dropped rows missing IP/ports: 0 (from 241911 -> 241911)
[clean] Step 2/3: columns with ±inf and/or missing values detected:
        ['DST_TO_SRC_SECOND_BYTES', 'SRC_TO_DST_SECOND_BYTES']
[clean] Step 3/3: filled 30018 missing numeric values with 0.
[clean] Done. Final shape: (241911, 55)


In [3]:
data.head()

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1528104433603,1528104433618,192.168.100.3,46560,199.7.83.42,53,17,5.0,56,1,...,0,0,0,0,0,0,0,0,0,Benign
1,1528103789264,1528103789264,192.168.100.149,51032,192.168.100.7,80,6,7.0,52,1,...,0,0,0,0,0,0,0,0,1,DDoS
2,1528103789265,1528103789265,192.168.100.149,51172,192.168.100.7,80,6,7.0,52,1,...,0,0,0,0,0,0,0,0,1,DDoS
3,1528103789266,1528103789266,192.168.100.149,51312,192.168.100.7,80,6,7.0,52,1,...,0,0,0,0,0,0,0,0,1,DDoS
4,1528103789267,1528103789267,192.168.100.149,51452,192.168.100.7,80,6,7.0,52,1,...,0,0,0,0,0,0,0,0,1,DDoS


In [4]:
# anonymize IP addresses by replacing them with random private IPs in the range
# To construct the network graph from the flow data, we mapped the original source IP addresses to randomly assigned
# IP addresses in the range from 172.16.0.1 to 172.31.0.1. The reason for this is the fact that in a lot of the NIDS datasets
# only a small number of IP addresses were used as the source of the attacks. The random mapping avoids
# the potential problem of the source IP addresses providing an unintentional label for attack traffic.
# NOTE: how to modify original randomizer code to temporally consistent flows (i.e., single src to dst should have matching dst to src flow)
#data['IPV4_SRC_ADDR'] = data.IPV4_SRC_ADDR.apply(lambda x: socket.inet_ntoa(struct.pack('>I', random.randint(0xac100001, 0xac1f0001))))

In [5]:
# convert IP addresses and ports to string type then concatenate them to form unique node identifiers
data['IPV4_SRC_ADDR'] = data.IPV4_SRC_ADDR.apply(str)
data['L4_SRC_PORT'] = data.L4_SRC_PORT.apply(str)
data['IPV4_DST_ADDR'] = data.IPV4_DST_ADDR.apply(str)
data['L4_DST_PORT'] = data.L4_DST_PORT.apply(str)

data['IPV4_SRC_ADDR'] = data['IPV4_SRC_ADDR'] + ':' + data['L4_SRC_PORT']
data['IPV4_DST_ADDR'] = data['IPV4_DST_ADDR'] + ':' + data['L4_DST_PORT']

data.drop(columns=['L4_SRC_PORT','L4_DST_PORT'],inplace=True)
data.info()
data.head()

<class 'pandas.core.frame.DataFrame'>
Index: 241911 entries, 0 to 241910
Data columns (total 53 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   FLOW_START_MILLISECONDS      241911 non-null  int64  
 1   FLOW_END_MILLISECONDS        241911 non-null  int64  
 2   IPV4_SRC_ADDR                241911 non-null  object 
 3   IPV4_DST_ADDR                241911 non-null  object 
 4   PROTOCOL                     241911 non-null  int64  
 5   L7_PROTO                     241911 non-null  float64
 6   IN_BYTES                     241911 non-null  int64  
 7   IN_PKTS                      241911 non-null  int64  
 8   OUT_BYTES                    241911 non-null  int64  
 9   OUT_PKTS                     241911 non-null  int64  
 10  TCP_FLAGS                    241911 non-null  int64  
 11  CLIENT_TCP_FLAGS             241911 non-null  int64  
 12  SERVER_TCP_FLAGS             241911 non-null  int64  
 13  FLOW

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,IPV4_DST_ADDR,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1528104433603,1528104433618,192.168.100.3:46560,199.7.83.42:53,17,5.0,56,1,1317,1,...,0,0,0,0,0,0,0,0,0,Benign
1,1528103789264,1528103789264,192.168.100.149:51032,192.168.100.7:80,6,7.0,52,1,0,0,...,0,0,0,0,0,0,0,0,1,DDoS
2,1528103789265,1528103789265,192.168.100.149:51172,192.168.100.7:80,6,7.0,52,1,0,0,...,0,0,0,0,0,0,0,0,1,DDoS
3,1528103789266,1528103789266,192.168.100.149:51312,192.168.100.7:80,6,7.0,52,1,0,0,...,0,0,0,0,0,0,0,0,1,DDoS
4,1528103789267,1528103789267,192.168.100.149:51452,192.168.100.7:80,6,7.0,52,1,0,0,...,0,0,0,0,0,0,0,0,1,DDoS


In [6]:
# we don't need the Label column as we are doing multiclass classification
data.drop(columns=['Label'],inplace = True)

In [7]:
data.rename(columns={"Attack": "label"},inplace = True)

## Chronological split of data to 60/30/10

In [8]:
from chronological_split import (
    make_chronological_split_indices,
    save_split_indices, load_split_indices,
    save_split_frames,
)

# df_clean = ...  # output of your data_cleaning.clean_nfunsw_nb15(df_raw)

# 1) Create indices (once)
train_idx, val_idx, test_idx, meta = make_chronological_split_indices(
    data,
    start_col="FLOW_START_MILLISECONDS",  # in ms
    end_col="FLOW_END_MILLISECONDS",
    dur_col="FLOW_DURATION_MILLISECONDS",
    train_ratio=0.60, val_ratio=0.30, test_ratio=0.10, #change to other ratios if needed
)

# 2) Persist indices for future runs
save_split_indices("artifacts/splits", train_idx, val_idx, test_idx, meta)

# (Optional) also dump the materialized split datasets now for reuse
save_split_frames(data, train_idx, val_idx, test_idx, out_dir="artifacts/splits", fmt="parquet")

# 3) In any future run (graph build, feature store, training), load the SAME indices:
train_idx2, val_idx2, test_idx2, meta2 = load_split_indices("artifacts/splits")

# Use them to subset clean dataframe (data) deterministically
df_train = data.loc[train_idx2]
df_val   = data.loc[val_idx2]
df_test  = data.loc[test_idx2]

[split] Chronological 60/30/10 with boundary safety
        TRAIN:   145149  t∈[1526344332972, 1528095906803]
        VAL:      72571  t∈(1528095906803, 1528099941218]
        TEST:     24191  t∈(1528099941218, 1529381825139]
[split] Saved indices → artifacts/splits/split_indices.npz and meta → artifacts/splits/meta.json
[split] Saved materialized splits to artifacts/splits (parquet).


## Label encoding

In [9]:
from label_mapping import fit_label_map, transform_labels, save_label_map, load_label_map, class_weights_from_train

# 1) Fit mapping on TRAIN ONLY
label2id = fit_label_map(df_train["label"], order="alpha")  # or order="freq"
save_label_map("artifacts/label_map.json", label2id)

# 2) Apply mapping to all splits (consistent)
label2id = load_label_map("artifacts/label_map.json")
y_train = transform_labels(df_train["label"], label2id)  # np.int64
y_val   = transform_labels(df_val["label"], label2id, unknown_policy="map_to_last")
y_test  = transform_labels(df_test["label"], label2id, unknown_policy="map_to_last")

# (Optional) sanity check: ensure no -1 slipped in
assert (y_train >= 0).all(), "Train split has unseen/missing labels."

# 3) Class weights for loss
weights = class_weights_from_train(y_train, num_classes=len(label2id))

## Numerical transformation
### Remove highly corelated features
This function visualise correlation between given features.
Also performs log1p, scaler, zero-variance drop, optional Spearman prune on numericals
Applies transformation to test and validation

In [10]:

from feature_numeric import fit_numeric_transform, transform_numeric

# Toggle correlation pruning
APPLY_CORR_PRUNE = True   # set to False to disable pruning and re-run preprocessing
# if you want to check sensitivity of final results to pruning make sure to run train model on best parameters only. and report results.

# 0) Columns to EXCLUDE here because they’re numeric-coded categoricals:
numeric_cats = [
    "PROTOCOL", "L7_PROTO",
    "ICMP_TYPE", "ICMP_IPV4_TYPE",
    # any other *_TYPE / *_ID style columns you want one-hot later
]

# 1) Correlation heatmap (pre-pruning) on TRAIN only
#viz_info = plot_numeric_corr_heatmap(
    #df_train,
    #exclude_numeric_categoricals=numeric_cats,
    #out_dir="artifacts/corr",
    #threshold=0.995,         # same ballpark as your pruning threshold
    #max_features=150,        # avoid unreadable giant plots
    #nonneg_frac=0.995,       # infer nonnegative cols for log1p
    #topk_pairs=100,
    #filename_prefix="spearman_corr_train"
#)
#print("[corr-viz]", viz_info)

# 2) TRAIN: fit numeric pipeline (log1p, scaler, zero-variance drop, optional Spearman prune)
Xnum_train, num_arts = fit_numeric_transform(
    df_train,
    exclude_numeric_categoricals=numeric_cats,
    scaler_type="standard",           # or "robust" if outliers are extreme
    apply_corr_prune=APPLY_CORR_PRUNE,
    corr_threshold=0.995,
    artifacts_dir="artifacts/numeric"
)
print(
    "[numeric] dimensions:",
    "before=", num_arts.dim_before_var,
    "after_variance=", num_arts.dim_after_var,
    "after_correlation=", num_arts.dim_after_corr,
    "Pruning=", APPLY_CORR_PRUNE,
)
print("[numeric] train shape:", Xnum_train.shape)

# 2) VAL/TEST: apply frozen transforms
Xnum_val, _  = transform_numeric(df_val,  artifacts_dir="artifacts/numeric")
Xnum_test, _ = transform_numeric(df_test, artifacts_dir="artifacts/numeric")
print("[numeric] val/test shapes:", Xnum_val.shape, Xnum_test.shape)

[numeric] dimensions: before= 45 after_variance= 45 after_correlation= 31 Pruning= True
[numeric] train shape: (145149, 31)
[numeric] val/test shapes: (72571, 31) (24191, 31)


## Categorical encoding
fits a OneHotEncoder on TRAIN only (sparse CSR, float32),
handles numeric-coded categoricals (e.g., PROTOCOL, L7_PROTO, *_TYPE, *_ID),
Optionally puts IPv4 ports in buckets, collapses rare categories (by min_freq or top_k) - not used here as we droped IPv4 ports after encoding it to unique hosts with IP addresses

persists artifacts (encoder + per-column metadata) for deterministic transforms,
transforms VAL/TEST,
returns CSR matrices ready to stack with numeric transformation.

In [11]:
from categorical_encoding import fit_categorical_transform, transform_categorical

# Choose categorical columns (include your numeric-coded categoricals here!)
cat_cols = [
    "PROTOCOL", "L7_PROTO",
    "ICMP_TYPE", "ICMP_IPV4_TYPE",
    "DNS_QUERY_TYPE", "DNS_QUERY_ID",
    "FTP_COMMAND_RET_CODE",
    # If you want ports as categories (after port bucketing):
    # "L4_SRC_PORT", "L4_DST_PORT", # we dropped ports after concatenating them to IP addresses
    # any other *_TYPE / *_ID style columns you want one-hot
]

# 1) TRAIN — fit encoder (optionally collapse rare cats)
Xcat_train, cat_arts = fit_categorical_transform(
    df_train,
    cat_cols=cat_cols,
    use_port_buckets=False,     # buckets ports into IANA groups to avoid huge dims
    min_freq=50,               # collapse categories with <50 occurrences to "__RARE__" (tune as needed)
    top_k=None,                # or e.g., top_k=20 to keep top-20 per column
    artifacts_dir="artifacts/categorical",
)
print("[categorical] train CSR shape:", Xcat_train.shape)

# 2) VAL/TEST — apply frozen encoder
Xcat_val  = transform_categorical(df_val,  artifacts_dir="artifacts/categorical")
Xcat_test = transform_categorical(df_test, artifacts_dir="artifacts/categorical")
print("[categorical] val/test CSR shapes:", Xcat_val.shape, Xcat_test.shape)


[categorical] train CSR shape: (145149, 21)
[categorical] val/test CSR shapes: (72571, 21) (24191, 21)


## Prepare feature store for mini-batch and other
What: Persist transformed features by split, not in the graph.
Why: Memory safety; fast random access by batch.

a) mini-batch with neighbrour sampling (k-layers)
b) random walk

In [12]:
from feature_store import build_feature_store

# We already have:
# df_train, df_val, df_test       (chronological splits)
# y_train, y_val, y_test          (int labels via label_mapping)
# train_idx, val_idx, test_idx    (global edge indices from persist_splits)

numeric_cats = [
    "PROTOCOL","L7_PROTO","ICMP_TYPE","ICMP_IPV4_TYPE",
    "DNS_QUERY_TYPE","DNS_QUERY_ID","FTP_COMMAND_RET_CODE",
    #"L4_SRC_PORT","L4_DST_PORT",  # if you bucket/one-hot ports - otherwise remove
]

cat_cols = numeric_cats  # plus any extra true categoricals if you have them

shapes = build_feature_store(
    df_train, df_val, df_test,
    y_train, y_val, y_test,
    train_idx, val_idx, test_idx,
    numeric_categoricals=numeric_cats,
    categorical_cols=cat_cols,
    out_dir="feature_store",
    numeric_artifacts_dir="artifacts/numeric",
    categorical_artifacts_dir="artifacts/categorical",
    use_port_buckets=False,
    rare_min_freq=50,
    rare_top_k=None,
    save_timestamps=True,
)
print(shapes)

[feature_store] train: n=145149 d_num=31 d_cat=21
[feature_store] val  : n=72571   d_num=31   d_cat=21
[feature_store] test : n=24191  d_num=31  d_cat=21
{'train': (145149, 31, 21), 'val': (72571, 31, 21), 'test': (24191, 31, 21)}


## Graph construction
What: Build DGL graph without edge features.
Why: Keep memory low; features come from the store per batch (fixed k-hop) or Random walk

No reverse duplication: we add exactly one directed edge per flow from the dataset; reverse flows (if present) appear naturally as their own edges later in time.

Alignment: g.edata[dgl.EID] stores the global edge IDs (original row indices) so your dataloader can fetch edge features from the feature store by ID in every batch.

Minimal memory: edge features live off-graph in the feature store; the graph only carries labels and timestamps.

In [13]:
from graph_build import build_light_graph_for_split

# These point to feature_store splits that were created earlier
split_dirs = {
    "train": "feature_store/train",
    "val":   "feature_store/val",
    "test":  "feature_store/test",
}

# df_train/df_val/df_test are the same dataframes used to build the store
ip2id = None

# build graphs for each split
g_train, ip2id = build_light_graph_for_split(
    df_train, split_dirs["train"],
    ip2id=ip2id, device="cpu",  # keep graphs on CPU; we can move blocks to GPU during training
    save_path="graphs/train.bin"
)

g_val, ip2id = build_light_graph_for_split(
    df_val,   split_dirs["val"],
    ip2id=ip2id, device="cpu",
    save_path="graphs/val.bin"
)

g_test, ip2id = build_light_graph_for_split(
    df_test,  split_dirs["test"],
    ip2id=ip2id, device="cpu",
    save_path="graphs/test.bin"
)

# print node and edge counts
print("Train graph: nodes =", g_train.num_nodes(), "edges =", g_train.num_edges())
print("Val graph:   nodes =", g_val.num_nodes(),   "edges =", g_val.num_edges())
print("Test graph:  nodes =", g_test.num_nodes(),  "edges =", g_test.num_edges())

Train graph: nodes = 117291 edges = 145149
Val graph:   nodes = 156350 edges = 72571
Test graph:  nodes = 167168 edges = 24191


In [14]:
# Create small graphs for hyperparameter testing purposes
# fraction of training edges to use with stratified sampling
import os
from sklearn.model_selection import StratifiedShuffleSplit

FRAC = 0.10

# =======================
# 1) SMALL TRAIN (stratified)
# =======================
# y_train is aligned with df_train = data.loc[train_idx2]
train_pos = np.arange(len(df_train))

sss_tr = StratifiedShuffleSplit(
    n_splits=1,
    train_size=FRAC,
    random_state=42,
)
small_train_pos, _ = next(sss_tr.split(train_pos, y_train))
small_train_pos = np.sort(small_train_pos)

# map positions in df_train back to global indices in `data`
train_idx2_arr = train_idx2.to_numpy() if hasattr(train_idx2, "to_numpy") else np.asarray(train_idx2)
train_sampled = np.sort(train_idx2_arr[small_train_pos])

train_idx_small = pd.Index(train_sampled)
print(
    f"Using {len(train_idx_small)} train edges out of {len(train_idx2)} "
    f"(~{FRAC*100:.1f}%) with stratified sampling"
)

df_train_small = data.loc[train_idx_small]
y_train_small  = y_train[small_train_pos]

# =======================
# 2) SMALL VAL (stratified)
# =======================
val_pos = np.arange(len(df_val))

sss_va = StratifiedShuffleSplit(
    n_splits=1,
    train_size=FRAC,
    random_state=43,
)
small_val_pos, _ = next(sss_va.split(val_pos, y_val))
small_val_pos = np.sort(small_val_pos)

val_idx2_arr = val_idx2.to_numpy() if hasattr(val_idx2, "to_numpy") else np.asarray(val_idx2)
val_sampled = np.sort(val_idx2_arr[small_val_pos])

val_idx_small = pd.Index(val_sampled)
print(
    f"Using {len(val_idx_small)} val edges out of {len(val_idx2)} "
    f"(~{FRAC*100:.1f}%) with stratified sampling"
)

df_val_small = data.loc[val_idx_small]
y_val_small  = y_val[small_val_pos]

# =======================
# 3) Small feature store (train+val small, test full)
# =======================
small_fs_dir = "hyper_tune_small"
os.makedirs(small_fs_dir, exist_ok=True)

numeric_cats = [
    "PROTOCOL","L7_PROTO","ICMP_TYPE","ICMP_IPV4_TYPE",
    "DNS_QUERY_TYPE","DNS_QUERY_ID","FTP_COMMAND_RET_CODE",
]
cat_cols = numeric_cats

shapes_small = build_feature_store(
    df_train_small, df_val_small, df_test,   # <-- small val here
    y_train_small, y_val_small, y_test,
    train_idx_small, val_idx_small, test_idx,
    numeric_categoricals=numeric_cats,
    categorical_cols=cat_cols,
    out_dir=small_fs_dir,
    numeric_artifacts_dir="artifacts/numeric",
    categorical_artifacts_dir="artifacts/categorical",
    use_port_buckets=False,
    rare_min_freq=50,
    rare_top_k=None,
    save_timestamps=True,
)
print("[small fs] shapes:", shapes_small)

# =======================
# 4) Small graphs (train+val small, test full)
# =======================
os.makedirs("graphs_small", exist_ok=True)

g_train_small, ip2id_small = build_light_graph_for_split(
    df_train_small, os.path.join(small_fs_dir, "train"),
    ip2id=None, device="cpu",
    save_path="graphs_small/train.bin",
)
g_val_small, ip2id_small = build_light_graph_for_split(
    df_val_small, os.path.join(small_fs_dir, "val"),   # <-- small val here
    ip2id=ip2id_small, device="cpu",
    save_path="graphs_small/val.bin",
)

print("Small train graph: nodes =", g_train_small.num_nodes(), "edges =", g_train_small.num_edges())
print("Small val graph:   nodes =", g_val_small.num_nodes(),   "edges =", g_val_small.num_edges())

Using 14514 train edges out of 145149 (~10.0%) with stratified sampling
Using 7257 val edges out of 72571 (~10.0%) with stratified sampling
[feature_store] train: n=14514 d_num=31 d_cat=15
[feature_store] val  : n=7257   d_num=31   d_cat=15
[feature_store] test : n=24191  d_num=31  d_cat=15
[small fs] shapes: {'train': (14514, 31, 15), 'val': (7257, 31, 15), 'test': (24191, 31, 15)}
Small train graph: nodes = 14808 edges = 14514
Small val graph:   nodes = 21639 edges = 7257


Sanity check

In [15]:
#Sanity check
import numpy as np, os
ytr = np.load("feature_store/train/y.npy"); yva = np.load("feature_store/val/y.npy")
print("train uniques:", np.unique(ytr), "size:", ytr.size)
print("val uniques  :", np.unique(yva), "size:", yva.size)
assert ytr.min() >= 0


train uniques: [0 1 2 3] size: 145149
val uniques  : [0 1] size: 72571


In [16]:
# graph sanity check
# one sample e_feat
import dgl, numpy as np

g = dgl.load_graphs("graphs/train.bin")[0][0]
sample_eids = g.edata[dgl.EID].cpu().numpy()
store_eids = np.load("feature_store/train/edge_indices.npy")
# start_idx_graph = np.where(sample_eids == 249625)[0][0]
# print("Sample EIDs from graph:", sample_eids[start_idx_graph:start_idx_graph+10])
print("Min EID:", store_eids.min(), "Max EID:", store_eids.max())
print("First 20 EIDs:", store_eids[:20])
print("Is 249625 in store_eids?", 249625 in store_eids)

print("graph_eids.shape:", sample_eids.shape, "min:", sample_eids.min(), "max:", sample_eids.max())
present_in_graph = np.any(sample_eids == 249625)
print("249625 in graph?", present_in_graph)

sample = g.edata[dgl.EID][:8].cpu().numpy()
from feature_store import fetch_edge_features
e = fetch_edge_features(sample, "feature_store/train")
print("edge feature dim:", e.shape[1])  # should equal edge_in (inferred)

Min EID: 1071 Max EID: 241881
First 20 EIDs: [239596 239597 239598 239599 239600 239601 239602 239603 239604 239605
 239606 239607 239608 239609 239610 239611 239612 239613 239614 239615]
Is 249625 in store_eids? False
graph_eids.shape: (145149,) min: 1071 max: 241881
249625 in graph? False
edge feature dim: 52


In [17]:
import os, numpy as np, dgl
# prove the split is “wired through” correctly
def assert_graph_equals_store(g, split_dir, name):
    s = np.load(os.path.join(split_dir, "edge_indices.npy")).astype(np.int64)
    ge = g.edata[dgl.EID].cpu().numpy().astype(np.int64)
    print(f"[{name}] graph={ge.size} store={s.size}")
    assert s.shape == ge.shape and np.array_equal(s, ge), f"{name}: graph EIDs != store edge_indices"

assert_graph_equals_store(g_train, "feature_store/train", "train")
assert_graph_equals_store(g_val,   "feature_store/val",   "val")
print("✅ graph == store for both splits")

[train] graph=145149 store=145149
[val] graph=72571 store=72571
✅ graph == store for both splits


In [18]:
# Check if we have node features and their shape (we dont so it should be None)
import dgl, torch
g = dgl.load_graphs("graphs/train.bin")[0][0]
print("has node feats:", "x" in g.ndata, "shape:" , (None if "x" not in g.ndata else tuple(g.ndata["x"].shape)))

has node feats: False shape: None


### Training

In [19]:
# training script for edge classification. Can be called from CLI or imported as a module
import copy, os, json, shutil
from types import SimpleNamespace

import torch
from train_edgecls_dbg import run_training

base_args = SimpleNamespace(
    feature_store="hyper_tune_small",   # use small FS for hyperparam tuning
    graphs_dir="graphs_small",          # use small graphs for hyperparam tuning
    split_train="train",
    split_val="val",
    hidden=128,         # 64, 128, 256
    layers=2,
    aggregator="mean",  # mean, pool, lstm, gcn
    edge_in=0,               # infer from sample
    edge_mlp_hidden=128,
    dropout=0.3,
    fanouts="25,15",    # 15,10 or 25,15 or 35,25
    batch_size=2048,    # 256, 512, 1024, 2048
    epochs=25,
    lr=3e-4,
    weight_decay=1e-4,
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_workers=0,
    seed=42,
    debug=False,
)
# hyperparameters
hiddens = [64, 128]
aggregators = ["mean"]
fanouts_grid = [(15,10), (25,15)]
dropouts = [0.2, 0.3, 0.4]
batch_sizes = [256, 512, 1024]

os.makedirs("artifacts/grid_models", exist_ok=True)
os.makedirs("artifacts/grid_logs", exist_ok=True)
results = []

for hidden in hiddens:
    for agg in aggregators:
        for bs in batch_sizes:
            for fan in fanouts_grid:
                for dr in dropouts:
                    args = copy.deepcopy(base_args)
                    args.hidden = hidden
                    args.aggregator = agg
                    args.dropout = dr
                    args.fanouts = f"{fan[0]},{fan[1]}"   # run_training expects string here
                    args.batch_size = bs
                    run_name = f"hid{hidden}_agg{agg}_fan{fan[0]}-{fan[1]}_drop{int(dr*100):02d}_bs{bs}"
                    print(f"=== RUN: {run_name} ===")
                    try:
                        res = run_training(args)   # returns dict with best_val_acc, edge_in, fanouts, history
                    except Exception as e:
                        print(f"[ERROR] run failed for {run_name}: {e}")
                        results.append({"run": run_name, "error": str(e)})
                        continue

                    best_acc = float(res.get("best_val_acc", -1.0))
                    # copy saved checkpoint (if run_training saved one) to unique path
                    src = "artifacts/best_edge_sage.pt"
                    dst = os.path.join("artifacts/grid_models", f"best_{run_name}.pt")
                    if os.path.exists(src):
                        shutil.copy(src, dst)
                    else:
                        dst = None
                    
                    # save per-epoch history for this run (acc, time etc.)
                    history = res.get("history", None)
                    if history is not None:
                        hist_path = os.path.join("artifacts/grid_logs", f"{run_name}_history.json")
                        with open(hist_path, "w", encoding="utf-8") as f:
                            json.dump(history, f, indent=2)
                    else:
                        hist_path = None

                    results.append({
                        "run": run_name,
                        "config": {
                            "hidden": hidden,
                            "aggregator": agg,
                            "fanouts": f"{fan[0]},{fan[1]}",
                            "dropout": dr,
                            "batch_size": bs,
                            "epochs": args.epochs,
                        },
                        "best_val_acc": best_acc,
                        "model_path": dst,
                        "train_return": res,
                        "history_path": hist_path,  # <--- pointer to per-epoch metrics if exists
                    })

                    # optional: remove the generic artifact so next run doesn't accidentally reuse it
                    if os.path.exists(src):
                        os.remove(src)

# persist results
with open("artifacts/grid_search_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

# print best trial
valid_results = [r for r in results if r.get("model_path")]
best = max(valid_results, key=lambda r: r["best_val_acc"]) if valid_results else None
print("BEST:", best)
# persist canonical model + params for other notebooks to load
os.makedirs("artifacts", exist_ok=True)
best_model_dst = "artifacts/best_edge_sage.pt"
best_params_path = "artifacts/best_params.json"

if best and best.get("model_path"):
    # copy the per-run checkpoint to the canonical filename expected by downstream code
    shutil.copy(best["model_path"], best_model_dst)

    # save minimal params so other notebooks can re-create the model/config
    params = {
        "run": best.get("run"),
        "config": best.get("config"),
        "best_val_acc": best.get("best_val_acc"),
        "model_path": best_model_dst,
    }
    with open(best_params_path, "w", encoding="utf-8") as f:
        json.dump(params, f, indent=2)

    print(f"Saved best model -> {best_model_dst} and params -> {best_params_path}")
else:
    if os.path.exists(best_model_dst):
        print(f"No successful new run — keeping existing {best_model_dst}")
    else:
        print("Warning: no best model available and artifacts/best_edge_sage.pt is missing")

=== RUN: hid64_aggmean_fan15-10_drop20_bs256 ===
2026-09-09 12:20:08,544 INFO [model] SAGE[0]: in=None out=None hidden=64 edge_in=46 in_node=0


c:\Users\nq100\miniconda3\DLLs\envs\te-g-sage\lib\site-packages\dgl\dataloading\dataloader.py:1149: DGLWarning: Dataloader CPU affinity opt is not enabled, consider switching it on (see enable_cpu_affinity() or CPU best practices for DGL [https://docs.dgl.ai/tutorials/cpu/cpu_best_practises.html])
  dgl_warning(


2026-09-09 12:20:11,047 INFO [timing] epoch: total_loss=16862.3860 total_time=1.0128s (avg batch 0.018s)
2026-09-09 12:20:11,405 INFO [timing] epoch: total_loss=15014.4944 total_time=0.3559s (avg batch 0.012s)
2026-09-09 12:20:11,407 INFO [epoch 01] train loss 1.1618 train acc 0.5126 | val loss 2.0690 val acc 0.0227
2026-09-09 12:20:12,423 INFO [timing] epoch: total_loss=13201.1016 total_time=1.0091s (avg batch 0.018s)
2026-09-09 12:20:12,772 INFO [timing] epoch: total_loss=23279.9560 total_time=0.3480s (avg batch 0.012s)
2026-09-09 12:20:12,773 INFO [epoch 02] train loss 0.9095 train acc 0.5691 | val loss 3.2079 val acc 0.0054
2026-09-09 12:20:13,753 INFO [timing] epoch: total_loss=12053.3836 total_time=0.9794s (avg batch 0.017s)
2026-09-09 12:20:14,100 INFO [timing] epoch: total_loss=25286.4979 total_time=0.3450s (avg batch 0.012s)
2026-09-09 12:20:14,101 INFO [epoch 03] train loss 0.8305 train acc 0.5594 | val loss 3.4844 val acc 0.0227
2026-09-09 12:20:15,059 INFO [timing] epoch: t

In [20]:
# run with best params
best_params_path = "artifacts/best_params.json"
if not os.path.exists(best_params_path):
    raise FileNotFoundError(best_params_path)
best = json.load(open(best_params_path, "r", encoding="utf-8"))
cfg = best.get("config", {})

# create a minimal template

template = SimpleNamespace(
    feature_store="feature_store",
    graphs_dir="graphs",
    split_train="train",
    split_val="val",
    hidden=128,
    layers=2,
    aggregator="mean",
    edge_in=0,
    edge_mlp_hidden=128,
    dropout=0.3,
    fanouts="25,15",
    batch_size=2048,
    epochs=25,
    lr=3e-4,
    weight_decay=1e-4,
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_workers=0,
    seed=42,
    debug=False,
)

args = copy.deepcopy(template)
# override template with saved config values
for k, v in cfg.items():
    # ensure fanouts is a string as expected by run_training
    if k == "fanouts" and not isinstance(v, str):
        v = str(v)
    setattr(args, k, v)

# set epochs to 25
args.epochs = 25

print("Running training with:", {k: getattr(args, k) for k in ("hidden","aggregator","fanouts","dropout","epochs", "batch_size")})
res = run_training(args)

# save metrics
history = res.get("history", None)
os.makedirs("artifacts", exist_ok=True)

full_hist_path = "artifacts/best_history.json"
if history is not None:
    with open(full_hist_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    print(f"Saved full-graph history -> {full_hist_path}")
else:
    print("Warning: run_training did not return 'history'; nothing to save.")

Running training with: {'hidden': 128, 'aggregator': 'mean', 'fanouts': '25,15', 'dropout': 0.2, 'epochs': 25, 'batch_size': 1024}
2026-09-09 12:34:47,627 INFO [model] SAGE[0]: in=None out=None hidden=128 edge_in=52 in_node=0
2026-09-09 12:34:53,690 INFO [timing] epoch: total_loss=120940.8291 total_time=6.0102s (avg batch 0.042s)
2026-09-09 12:34:55,134 INFO [timing] epoch: total_loss=87815.0219 total_time=1.4415s (avg batch 0.020s)
2026-09-09 12:34:55,134 INFO [epoch 01] train loss 0.8332 train acc 0.5897 | val loss 1.2101 val acc 0.6773
2026-09-09 12:35:00,827 INFO [timing] epoch: total_loss=78929.5948 total_time=5.6911s (avg batch 0.040s)
2026-09-09 12:35:02,230 INFO [timing] epoch: total_loss=98810.8632 total_time=1.4017s (avg batch 0.020s)
2026-09-09 12:35:02,230 INFO [epoch 02] train loss 0.5438 train acc 0.6479 | val loss 1.3616 val acc 0.6772
2026-09-09 12:35:07,917 INFO [timing] epoch: total_loss=67115.4056 total_time=5.6864s (avg batch 0.040s)
2026-09-09 12:35:09,338 INFO [ti

In [21]:
print("Train:")
print(df_train["label"].value_counts())
print("\nVal:")
print(df_val["label"].value_counts())
print("\nTest:")
print(df_test["label"].value_counts())

Train:
label
DoS               114776
Reconnaissance     24220
DDoS                5444
Benign               709
Name: count, dtype: int64

Val:
label
DDoS      72556
Benign       15
Name: count, dtype: int64

Test:
label
DDoS      24153
Theft        26
Benign       12
Name: count, dtype: int64
